# 시간여행TV식 테마 화재경보기 v2

**구조 좋은 소형주 × 테마 점화 × 거래대금 최초 증가**를 결합합니다.

- KRX/FinanceDataReader: 상장종목·가격
- Npay 증권 테마 페이지: 테마 수익률·확산도·구성종목
- OpenDART(선택): 최대주주·소액주주·최근 CB/BW/유증 결정

최종점수는 구조 35% × 종목수급 35% × 테마점화 30%의 가중 기하평균입니다. 어느 한 축이 약하면 점수가 눌립니다.

`DART_API_KEY`를 비우면 Lite 모드, 입력하면 Full 모드입니다.

주의: 최근 CB/BW/유증 결정은 실제 미전환 잔액이 아니라 **희석 위험 프록시**입니다. 시장경보·거래정지·공시 원문은 매수 전 별도 확인하세요.

In [ ]:
!pip -q install finance-datareader beautifulsoup4 lxml pandas numpy requests tqdm


## 실행
아래 셀 맨 위의 `DART_API_KEY`와 범위 설정을 필요에 맞게 수정한 뒤 실행하세요.


In [ ]:
import os
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
import io, zipfile, xml.etree.ElementTree as ET
import re, time, warnings, math
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import FinanceDataReader as fdr
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings('ignore')

# ================= USER SETTINGS =================
DART_API_KEY = os.environ.get("DART_API_KEY", "").strip()
LOWFLOAT_WARNINGS = []
DART_EFFECTIVE_MODE = "FULL" if DART_API_KEY else "LITE"
MIN_MCAP_EOK = 500
MAX_MCAP_EOK = 5000
MIN_PRICE = 1000
MAX_LISTED_SHARES = 80_000_000
MIN_TODAY_AMOUNT_EOK = 1.0
MAX_TODAY_AMOUNT_EOK = 1000.0
THEME_MAX_PAGES = 12
MAX_THEMES_TO_DETAIL = 140
MIN_THEME_MEMBERS = 3
MIN_THEME_CATALOG_SCORE = 18
HISTORY_CALENDAR_DAYS = 210
MAX_STOCKS_FOR_HISTORY = 180
PRICE_WORKERS = 10
THEME_WORKERS = 6
DART_TOP_N = 60
DART_OVERHANG_LOOKBACK_DAYS = 730

NOW = datetime.now(ZoneInfo('Asia/Seoul'))
TODAY = NOW.date()
START_DATE = (TODAY - timedelta(days=HISTORY_CALENDAR_DAYS)).isoformat()
NAVER_HEADERS = {'User-Agent':'Mozilla/5.0 Chrome/151 Safari/537.36','Referer':'https://finance.naver.com/'}
print('KST:', NOW.strftime('%Y-%m-%d %H:%M:%S'), '| DART:', 'FULL' if DART_API_KEY.strip() else 'LITE')

def clip01(x): return np.clip(x,0.0,1.0)
def pnum(x):
    if x is None: return np.nan
    s=str(x).replace(',','').replace('%','').replace('+','').strip()
    try: return float(s)
    except: return np.nan

def pint(x):
    v=pnum(x); return int(v) if pd.notna(v) else 0

def norm_code(x):
    m=re.search(r'(\d{6})',str(x)); return m.group(1) if m else str(x).zfill(6)

def safe_get(url,params=None,retries=3):
    last=None
    for i in range(retries):
        try:
            r=requests.get(url,params=params,headers=NAVER_HEADERS,timeout=20); r.raise_for_status(); return r
        except Exception as e:
            last=e; time.sleep(.7*(i+1))
    raise last

def gscore(vals,weights):
    v=np.clip(np.array(vals,dtype=float),1,100)/100
    w=np.array(weights,dtype=float); w=w/w.sum()
    return float(np.exp(np.sum(w*np.log(v)))*100)

# ================= KRX UNIVERSE =================
krx=fdr.StockListing('KRX').copy()
if 'Code' not in krx.columns and 'Symbol' in krx.columns: krx=krx.rename(columns={'Symbol':'Code'})
krx['Code']=krx['Code'].map(norm_code)
for c in ['Close','Volume','Amount','Marcap','Stocks']:
    if c in krx.columns: krx[c]=pd.to_numeric(krx[c],errors='coerce')
krx=krx[krx['Market'].isin(['KOSPI','KOSDAQ'])].copy()
krx=krx[~krx['Name'].astype(str).str.contains(r'(스팩|SPAC|리츠|ETF|ETN|인버스|레버리지)',case=False,regex=True,na=False)]
krx=krx[~krx['Name'].astype(str).str.contains(r'(우$|우B$|우C$|우선주)',regex=True,na=False)]
try:
    ad=fdr.StockListing('KRX-ADMINISTRATIVE').copy(); cc='Code' if 'Code' in ad.columns else 'Symbol'
    admin=set(ad[cc].astype(str).map(norm_code)); krx=krx[~krx['Code'].isin(admin)]
except: pass
if 'Dept' in krx.columns:
    krx=krx[~krx['Dept'].astype(str).str.contains(r'(관리|환기|정리매매|거래정지)',regex=True,na=False)]
krx['시총_억원']=krx['Marcap']/1e8
krx['상장주식수_백만주']=krx['Stocks']/1e6
krx['당일거래대금_억원']=krx['Amount']/1e8
krx['당일회전율_%']=np.where(krx['Stocks']>0,krx['Volume']/krx['Stocks']*100,np.nan)
universe=krx[
    krx['시총_억원'].between(MIN_MCAP_EOK,MAX_MCAP_EOK)&
    (krx['Close']>=MIN_PRICE)&(krx['Stocks']<=MAX_LISTED_SHARES)&
    krx['당일거래대금_억원'].between(MIN_TODAY_AMOUNT_EOK,MAX_TODAY_AMOUNT_EOK)
].copy()
print('1차 유니버스:',len(universe))

def theme_score(day,d3,up,flat,down):
    total=up+flat+down; breadth=up/total if total else 0
    s=30*clip01(max(day,0)/5)+35*clip01((breadth-.45)/.40)+20*clip01(max(d3,0)/7)
    s+=10*clip01((day-d3/3)/3)+5*clip01((total-2)/15)
    if day>8: s-=8
    if d3>15: s-=min(20,(d3-15)*1.5)
    return round(float(np.clip(s,0,100)),1),round(breadth*100,1)

def scrape_theme_catalog():
    rows=[]; seen=set()
    for page in range(1,THEME_MAX_PAGES+1):
        soup=BeautifulSoup(safe_get('https://finance.naver.com/sise/theme.naver',{'page':page}).text,'lxml')
        found=0
        for tr in soup.select('table.type_1 tr'):
            a=tr.select_one('a[href*="sise_group_detail.naver"][href*="type=theme"]')
            if not a: continue
            m=re.search(r'(?:\?|&)no=(\d+)',a.get('href',''))
            if not m or m.group(1) in seen: continue
            t=[td.get_text(' ',strip=True) for td in tr.find_all('td')]
            if len(t)<6: continue
            day,d3=pnum(t[1]),pnum(t[2]); up,flat,down=pint(t[3]),pint(t[4]),pint(t[5])
            if pd.isna(day): continue
            if pd.isna(d3): d3=0
            sc,br=theme_score(day,d3,up,flat,down)
            rows.append([m.group(1),a.get_text(' ',strip=True),day,d3,up,flat,down,br,sc]); seen.add(m.group(1)); found+=1
        if found==0: break
        time.sleep(.12)
    return pd.DataFrame(rows,columns=['ThemeNo','Theme','테마_오늘_%','테마_최근3일_%','상승종목','보합종목','하락종목','테마_확산도_%','테마점화점수'])

def scrape_members(no,name):
    soup=BeautifulSoup(safe_get('https://finance.naver.com/sise/sise_group_detail.naver',{'type':'theme','no':no}).text,'lxml')
    out=[]
    for tr in soup.select('table.type_5 tr'):
        a=tr.select_one('a[href*="/item/main.naver?code="]')
        if not a: continue
        m=re.search(r'code=(\d{6})',a.get('href',''))
        if not m: continue
        nums=[td.get_text(' ',strip=True) for td in tr.select('td.number')]
        pct=pnum(nums[2]) if len(nums)>=3 else np.nan
        ntd=a.find_parent('td'); reason=''
        if ntd:
            txt=ntd.get_text(' ',strip=True); nm=a.get_text(' ',strip=True)
            txt=re.sub(r'^'+re.escape(nm)+r'\s*\*?','',txt).strip(); txt=re.sub(r'^테마 편입 사유\s*','',txt).strip(); reason=txt[:500]
        out.append([str(no),name,m.group(1),a.get_text(' ',strip=True),pct,reason])
    return out

themes=scrape_theme_catalog()
if themes.empty:
    LOWFLOAT_WARNINGS.append('Naver 테마 목록을 받지 못했습니다.')
themes['테마종목수']=themes['상승종목']+themes['보합종목']+themes['하락종목']
active=themes[(themes['테마종목수']>=MIN_THEME_MEMBERS)&(themes['테마점화점수']>=MIN_THEME_CATALOG_SCORE)].sort_values('테마점화점수',ascending=False).head(MAX_THEMES_TO_DETAIL)
print('활성 테마:',len(active)); display(active.head(30))

member_rows=[]
with ThreadPoolExecutor(max_workers=THEME_WORKERS) as ex:
    futs=[ex.submit(scrape_members,r.ThemeNo,r.Theme) for r in active.itertuples(index=False)]
    for f in tqdm(as_completed(futs),total=len(futs),desc='테마구성'):
        try: member_rows.extend(f.result())
        except: pass
members=pd.DataFrame(member_rows,columns=['ThemeNo','Theme','Code','Name_theme','테마내_당일등락_%','테마편입사유'])
if members.empty and not active.empty:
    LOWFLOAT_WARNINGS.append('활성 테마는 있었지만 구성종목을 받지 못했습니다.')
members=members.merge(active[['ThemeNo','Theme','테마_오늘_%','테마_최근3일_%','테마_확산도_%','테마종목수','테마점화점수']],on=['ThemeNo','Theme'],how='left')
members=members[members['Code'].isin(set(universe['Code']))].copy()
members['테마상대강도_%']=members['테마내_당일등락_%']-members['테마_오늘_%']
best=members.sort_values(['테마점화점수','테마상대강도_%'],ascending=[False,False]).groupby('Code',as_index=False).head(1).copy()
allthemes=members.sort_values(['Code','테마점화점수'],ascending=[True,False]).groupby('Code')['Theme'].agg(lambda s:' | '.join(pd.unique(s.astype(str)))[:1000]).rename('활성테마목록')
best=best.merge(allthemes,on='Code',how='left')
candidate=universe.merge(best[['Code','Theme','테마_오늘_%','테마_최근3일_%','테마_확산도_%','테마종목수','테마점화점수','테마내_당일등락_%','테마상대강도_%','테마편입사유','활성테마목록']],on='Code',how='inner')
candidate=candidate.sort_values(['테마점화점수','시총_억원'],ascending=[False,True]).head(MAX_STOCKS_FOR_HISTORY).copy()
print('가격이력 후보:',len(candidate))

# ================= PRICE / MONEY FLOW =================
def hist_feature(code):
    try:
        df=fdr.DataReader(f'KRX:{code}',START_DATE)
        if df is None or len(df)<25: return {'Code':code,'history_ok':False}
        df=df.copy().sort_index()
        for c in ['Open','High','Low','Close','Volume','Amount']:
            if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
        if 'Amount' not in df.columns: df['Amount']=df['Close']*df['Volume']
        else: df['Amount']=df['Amount'].fillna(df['Close']*df['Volume'])
        df=df.dropna(subset=['Close','Volume','Amount'])
        if len(df)<25: return {'Code':code,'history_ok':False}
        def ret(n): return (df['Close'].iloc[-1]/df['Close'].iloc[-1-n]-1)*100 if len(df)>n else np.nan
        p20=df.iloc[-21:-1]; p5=df.iloc[-6:-1]
        at=float(df['Amount'].iloc[-1]); a20=float(p20['Amount'].mean()); a5=float(p5['Amount'].mean())
        v20=float(p20['Volume'].mean()); vt=float(df['Volume'].iloc[-1])
        h120=float((df['High'] if 'High' in df.columns else df['Close']).tail(120).max()); close=float(df['Close'].iloc[-1])
        return {'Code':code,'history_ok':True,'ret1_%':ret(1),'ret3_%':ret(3),'ret5_%':ret(5),'ret20_%':ret(20),
                '거래대금20배':at/a20 if a20>0 else np.nan,'거래대금5배':at/a5 if a5>0 else np.nan,
                '거래량20배':vt/v20 if v20>0 else np.nan,'120일고점대비_%':(close/h120-1)*100 if h120>0 else np.nan,
                '히스토리_당일거래대금_억원':at/1e8}
    except: return {'Code':code,'history_ok':False}

hrs=[]
with ThreadPoolExecutor(max_workers=PRICE_WORKERS) as ex:
    futs={ex.submit(hist_feature,c):c for c in candidate['Code']}
    for f in tqdm(as_completed(futs),total=len(futs),desc='가격/거래대금'): hrs.append(f.result())
_hist_cols=['Code','history_ok','ret1_%','ret3_%','ret5_%','ret20_%','거래대금20배','거래대금5배','거래량20배','120일고점대비_%','히스토리_당일거래대금_억원']
hist=pd.DataFrame(hrs)
if hist.empty:
    hist=pd.DataFrame(columns=_hist_cols)
else:
    for _c in _hist_cols:
        if _c not in hist.columns:
            hist[_c]=np.nan
candidate=candidate.merge(hist,on='Code',how='left')
if 'history_ok' not in candidate.columns:
    candidate['history_ok']=False
candidate=candidate[candidate['history_ok']==True].copy()
if not best.empty and candidate.empty:
    LOWFLOAT_WARNINGS.append('테마 후보는 있었지만 유효한 가격 이력을 확보하지 못했습니다.')

def amount_sc(x):
    if pd.isna(x) or x<1.2:return 0
    if x<1.5:return 5
    if x<2:return 12
    if x<3:return 20
    if x<=8:return 30
    if x<=12:return 23
    return 15

def r1_sc(x):
    if pd.isna(x) or x<-3:return 0
    if x<0:return 3
    if x<2:return 7
    if x<=8:return 15
    if x<=15:return 12
    if x<=22:return 6
    return 0

def unext(r5,r20):
    s=20
    if pd.notna(r5) and r5>20:s-=min(10,(r5-20)*.5)
    if pd.notna(r20) and r20>35:s-=min(15,(r20-35)*.35)
    if pd.notna(r5) and r5<-15:s-=5
    return float(np.clip(s,0,20))

def rel_sc(x):
    if pd.isna(x) or x<-3:return 0
    if x<0:return 3
    if x<=8:return 12
    if x<=15:return 9
    return 5

def overhead_sc(x):
    if pd.isna(x):return 3
    if x>=-5:return 10
    if x>=-15:return 8
    if x>=-30:return 5
    if x>=-50:return 2
    return 0

def turn_sc(a,m):
    if pd.isna(a) or pd.isna(m) or m<=0:return 0
    x=a/m*100
    if x<.2:return 0
    if x<.5:return 3
    if x<2:return 7
    if x<=12:return 13
    if x<=30:return 10
    return 5

candidate['수급_거래대금점수']=candidate['거래대금20배'].apply(amount_sc)
candidate['수급_당일가격점수']=candidate['ret1_%'].apply(r1_sc)
candidate['수급_미급등점수']=[unext(a,b) for a,b in zip(candidate['ret5_%'],candidate['ret20_%'])]
candidate['수급_상대강도점수']=candidate['테마상대강도_%'].apply(rel_sc)
candidate['수급_매물대점수']=candidate['120일고점대비_%'].apply(overhead_sc)
candidate['수급_회전점수']=[turn_sc(a,m) for a,m in zip(candidate['당일거래대금_억원'],candidate['시총_억원'])]
candidate['종목수급점화점수']=(candidate['수급_거래대금점수']+candidate['수급_당일가격점수']+candidate['수급_미급등점수']+candidate['수급_상대강도점수']+candidate['수급_매물대점수']+candidate['수급_회전점수']).clip(0,100).round(1)

# ================= DART =================
def corp_map(api):
    r=requests.get('https://opendart.fss.or.kr/api/corpCode.xml',params={'crtfc_key':api},timeout=30); r.raise_for_status()
    z=zipfile.ZipFile(io.BytesIO(r.content)); root=ET.fromstring(z.read('CORPCODE.xml')); rows=[]
    for x in root.findall('list'):
        sc=(x.findtext('stock_code') or '').strip()
        if sc: rows.append([(x.findtext('corp_code') or '').strip(),norm_code(sc)])
    return pd.DataFrame(rows,columns=['corp_code','Code']).drop_duplicates('Code')

def reports():
    y,m=NOW.year,NOW.month; a=[]
    if m>=11:a.append((y,'11014','3Q'))
    if m>=8:a.append((y,'11012','2Q'))
    if m>=5:a.append((y,'11013','1Q'))
    a.append((y-1,'11011','FY')); return a

def dart(endpoint,api,**p):
    r=requests.get(f'https://opendart.fss.or.kr/api/{endpoint}.json',params={'crtfc_key':api,**p},timeout=18); r.raise_for_status(); return r.json()

def holder_pct(rows):
    vals=[]; totals=[]
    for z in rows or []:
        kind=str(z.get('stock_knd','') or '')
        if kind and '보통' not in kind: continue
        v=pnum(z.get('trmend_posesn_stock_qota_rt')); nm=str(z.get('nm','') or '').strip(); rel=str(z.get('relate','') or '').strip()
        if pd.isna(v):continue
        if nm in ('계','합계','총계') or rel in ('계','합계','총계'):totals.append(v)
        else:vals.append(v)
    if totals:return min(max(totals),100)
    if vals:return min(sum(vals),100)
    return np.nan

def holder_minority(api,corp):
    out={'최대주주등지분율_%':np.nan,'소액주주보유비율_%':np.nan,'DART기준보고서':None}
    for y,rc,label in reports():
        h=dart('hyslrSttus',api,corp_code=corp,bsns_year=str(y),reprt_code=rc); m=dart('mrhlSttus',api,corp_code=corp,bsns_year=str(y),reprt_code=rc)
        hv=holder_pct(h.get('list',[])) if h.get('status')=='000' else np.nan; mv=np.nan
        if m.get('status')=='000':
            arr=m.get('list',[])
            for z in arr:
                if '소액' in str(z.get('se','') or ''): mv=pnum(z.get('hold_stock_rate')); break
            if pd.isna(mv) and arr: mv=pnum(arr[0].get('hold_stock_rate'))
        if pd.notna(hv):out['최대주주등지분율_%']=hv
        if pd.notna(mv):out['소액주주보유비율_%']=mv
        if pd.notna(hv) or pd.notna(mv):out['DART기준보고서']=f'{y}-{label}'
        if pd.notna(hv) and pd.notna(mv):break
    return out

def dilution(api,corp,listed):
    b=(TODAY-timedelta(days=DART_OVERHANG_LOOKBACK_DAYS)).strftime('%Y%m%d'); e=TODAY.strftime('%Y%m%d')
    cb=bw=ri=0; potential=0.; amount=0.
    try:
        j=dart('cvbdIsDecsn',api,corp_code=corp,bgn_de=b,end_de=e)
        if j.get('status')=='000':
            for z in j.get('list',[]):
                cb+=1; a=pnum(z.get('bd_fta')); p=pnum(z.get('cv_prc'))
                if pd.notna(a):amount+=a
                if pd.notna(a) and pd.notna(p) and p>0:potential+=a/p
    except:pass
    try:
        j=dart('bdwtIsDecsn',api,corp_code=corp,bgn_de=b,end_de=e)
        if j.get('status')=='000':
            for z in j.get('list',[]):
                bw+=1; a=pnum(z.get('bd_fta')); n=pnum(z.get('nstk_isstk_cnt'))
                if pd.notna(a):amount+=a
                if pd.notna(n):potential+=n
    except:pass
    try:
        j=dart('piicDecsn',api,corp_code=corp,bgn_de=b,end_de=e)
        if j.get('status')=='000':
            for z in j.get('list',[]):
                ri+=1; n=pnum(z.get('nstk_ostk_cnt'))
                if pd.notna(n):potential+=n
    except:pass
    return {'최근CB결정수':cb,'최근BW결정수':bw,'최근유증결정수':ri,'최근CB_BW발행결정액_억원':amount/1e8,'희석위험프록시_%':potential/listed*100 if listed and listed>0 else np.nan}

def base_struct(r):
    m=25*clip01((MAX_MCAP_EOK-r['시총_억원'])/(MAX_MCAP_EOK-MIN_MCAP_EOK)); sh=20*clip01((MAX_LISTED_SHARES-r['Stocks'])/MAX_LISTED_SHARES); a=r['당일거래대금_억원']; liq=2 if a<2 else 10 if a<=100 else 7 if a<=300 else 4
    return float(np.clip(m+sh+liq,0,55))

candidate['구조기초점수_55']=candidate.apply(base_struct,axis=1)
_DART_COLS=['최대주주등지분율_%','소액주주보유비율_%','추정유통시총_억원','희석위험프록시_%','최근CB결정수','최근BW결정수','최근유증결정수']

def _apply_lite_structure(df, label='LITE'):
    df['구조점수']=(df['구조기초점수_55']/55*100).round(1)
    df['구조데이터신뢰도']=label
    for _c in _DART_COLS:
        if _c not in df.columns:
            df[_c]=np.nan
    return df

if not DART_API_KEY.strip():
    candidate=_apply_lite_structure(candidate,'LITE')
elif candidate.empty:
    # 후보가 0개인 날에는 DART를 호출할 필요가 없다. 이것은 오류가 아니다.
    candidate=_apply_lite_structure(candidate,'FULL_NO_CANDIDATES')
else:
    try:
        api=DART_API_KEY.strip(); cm=corp_map(api)
        candidate['임시결합점수']=.45*candidate['구조기초점수_55']/55*100+.30*candidate['테마점화점수']+.25*candidate['종목수급점화점수']
        dt=candidate.sort_values('임시결합점수',ascending=False).head(DART_TOP_N).merge(cm,on='Code',how='left'); out=[]
        for r in tqdm(dt.itertuples(index=False),total=len(dt),desc='DART 정밀검사'):
            d={'Code':r.Code}; corp=getattr(r,'corp_code',None)
            if corp and not pd.isna(corp):
                try:d.update(holder_minority(api,str(corp)))
                except Exception as _e: pass
                try:d.update(dilution(api,str(corp),r.Stocks))
                except Exception as _e: pass
            out.append(d); time.sleep(.04)
        dart_df=pd.DataFrame(out)
        if dart_df.empty:
            dart_df=pd.DataFrame(columns=['Code']+_DART_COLS)
        candidate=candidate.merge(dart_df,on='Code',how='left')
        for _c in _DART_COLS:
            if _c not in candidate.columns:
                candidate[_c]=np.nan
        candidate['추정유통가능비율_%']=(100-candidate['최대주주등지분율_%']).clip(0,100)
        candidate['추정유통시총_억원']=candidate['시총_억원']*candidate['추정유통가능비율_%']/100
        def full_struct(r):
            s=r['구조기초점수_55']/55*35; h=r.get('최대주주등지분율_%',np.nan); mi=r.get('소액주주보유비율_%',np.nan); fc=r.get('추정유통시총_억원',np.nan); di=r.get('희석위험프록시_%',np.nan)
            if pd.notna(h):s+=20*clip01((h-20)/50)
            if pd.notna(mi):s+=15*clip01((70-mi)/55)
            if pd.notna(fc):s+=20*clip01((2000-fc)/1800)
            ov=10
            if pd.notna(di): ov=0 if di>30 else 3 if di>15 else 6 if di>7 else 10
            ev=sum(int(r.get(c,0)) for c in ['최근CB결정수','최근BW결정수','최근유증결정수'] if pd.notna(r.get(c,np.nan)))
            if ev>=4:ov=min(ov,2)
            elif ev>=2:ov=min(ov,5)
            return round(float(np.clip(s+ov,0,100)),1)
        candidate['구조점수']=candidate.apply(full_struct,axis=1)
        candidate['구조데이터신뢰도']=np.where(candidate['최대주주등지분율_%'].notna(),'FULL','PARTIAL')
        if candidate['최대주주등지분율_%'].notna().sum()==0:
            DART_EFFECTIVE_MODE='PARTIAL'
            LOWFLOAT_WARNINGS.append('DART 키는 있으나 최대주주 데이터를 확보하지 못해 구조점수 신뢰도가 낮습니다.')
    except Exception as _e:
        DART_EFFECTIVE_MODE='PARTIAL'
        LOWFLOAT_WARNINGS.append(f'DART 정밀검사 실패로 Lite 구조점수 사용: {type(_e).__name__}: {_e}')
        candidate=_apply_lite_structure(candidate,'DART_FALLBACK')

# ================= FINAL SCORE / ALERT =================
def final_score(r):
    s=gscore([r['구조점수'],r['종목수급점화점수'],r['테마점화점수']],[.35,.35,.30]); p=0
    if pd.notna(r.get('ret1_%')) and r['ret1_%']>20:p+=8
    if pd.notna(r.get('ret5_%')) and r['ret5_%']>30:p+=8
    if pd.notna(r.get('ret20_%')) and r['ret20_%']>60:p+=10
    if pd.notna(r.get('거래대금20배')) and r['거래대금20배']>15:p+=5
    if pd.notna(r.get('테마_최근3일_%')) and r['테마_최근3일_%']>18:p+=5
    if pd.notna(r.get('희석위험프록시_%')) and r['희석위험프록시_%']>20:p+=8
    return round(max(0,s-p),1)

def fire(r):
    ar, r1, r5, r20 = r['거래대금20배'],r['ret1_%'],r['ret5_%'],r['ret20_%']
    th, br = r['테마점화점수'],r['테마_확산도_%']
    if (pd.notna(r1) and r1>20) or (pd.notna(r5) and r5>30) or (pd.notna(r20) and r20>60):return 'LATE_과열주의'
    if pd.notna(ar) and 3<=ar<=10 and 2<=r1<=15 and th>=55 and br>=58:return 'IGNITION_점화'
    if pd.notna(ar) and 1.5<=ar<3.5 and -1<=r1<=8 and th>=45 and br>=52:return 'PRE_점화직전'
    if th>=45 and pd.notna(ar) and ar>=1.2:return 'WATCH_감시'
    return 'COLD'

candidate['최종점수']=candidate.apply(final_score,axis=1); candidate['화재경보']=candidate.apply(fire,axis=1)
def grade(r):
    if r['최종점수']>=75 and r['화재경보'] in ('PRE_점화직전','IGNITION_점화'):return 'S'
    if r['최종점수']>=65:return 'A'
    if r['최종점수']>=55:return 'B'
    return 'WATCH'
candidate['등급']=candidate.apply(grade,axis=1)
order={'IGNITION_점화':0,'PRE_점화직전':1,'WATCH_감시':2,'COLD':3,'LATE_과열주의':4}
candidate['_o']=candidate['화재경보'].map(order).fillna(9)
result=candidate.sort_values(['_o','최종점수','테마점화점수'],ascending=[True,False,False]).drop(columns=['_o'])
hot=result[result['화재경보'].isin(['PRE_점화직전','IGNITION_점화'])].copy()

def explain(r):
    x=[]
    if pd.notna(r.get('거래대금20배')):x.append(f"거래대금 {r['거래대금20배']:.1f}배")
    if pd.notna(r.get('테마_확산도_%')):x.append(f"테마확산 {r['테마_확산도_%']:.0f}%")
    if pd.notna(r.get('ret1_%')):x.append(f"당일 {r['ret1_%']:+.1f}%")
    if pd.notna(r.get('ret20_%')):x.append(f"20일 {r['ret20_%']:+.1f}%")
    if pd.notna(r.get('추정유통시총_억원')):x.append(f"유통시총≈{r['추정유통시총_억원']:.0f}억")
    if pd.notna(r.get('최대주주등지분율_%')):x.append(f"최대주주등 {r['최대주주등지분율_%']:.1f}%")
    if pd.notna(r.get('희석위험프록시_%')):x.append(f"희석프록시 {r['희석위험프록시_%']:.1f}%")
    return ' / '.join(x)
result['선정이유']=result.apply(explain,axis=1)

show=['등급','화재경보','Code','Name','Market','Theme','활성테마목록','최종점수','구조점수','종목수급점화점수','테마점화점수','시총_억원','추정유통시총_억원','최대주주등지분율_%','소액주주보유비율_%','거래대금20배','당일거래대금_억원','ret1_%','ret5_%','ret20_%','테마_오늘_%','테마_최근3일_%','테마_확산도_%','테마상대강도_%','120일고점대비_%','희석위험프록시_%','최근CB결정수','최근BW결정수','최근유증결정수','구조데이터신뢰도','선정이유']
show=[c for c in show if c in result.columns]
print('\n=== TOP RESULTS ==='); display(result[show].head(60))
print('\n=== PRE / IGNITION ==='); display(hot[[c for c in show if c in hot.columns]].head(30)); print('HOT:',len(hot))

tag=TODAY.strftime('%Y%m%d')
result.to_csv(f'theme_fire_alarm_all_{tag}.csv',index=False,encoding='utf-8-sig')
hot.to_csv(f'theme_fire_alarm_hot_{tag}.csv',index=False,encoding='utf-8-sig')
active.to_csv(f'theme_fire_alarm_themes_{tag}.csv',index=False,encoding='utf-8-sig')
print(f'CSV saved: theme_fire_alarm_all_{tag}.csv / hot / themes')


## 신호 해석
- **PRE_점화직전**: 거래대금 약 1.5~3.5배, 가격은 아직 덜 오른 상태, 테마 확산 시작
- **IGNITION_점화**: 거래대금 3~10배, 종목 +2~15%, 테마 확산도 58% 이상
- **LATE_과열주의**: 당일/5일/20일 급등으로 신규 진입 관점에서 후순위

결과 CSV 3개(전체/핫/테마)가 Colab 작업폴더에 자동 저장됩니다.


In [ ]:
# Telegram automation structured summary
import json as _json

def _tg_scalar(v):
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    if hasattr(v, "item"):
        try:
            return v.item()
        except Exception:
            pass
    return v

try:
    if "result" not in globals():
        raise RuntimeError("메인 스크리너 결과(result)가 생성되지 않았습니다. 위 셀의 실제 오류를 확인하세요.")
    _counts = result["화재경보"].value_counts().to_dict() if "화재경보" in result.columns else {}
    _hot = result[result["화재경보"].isin(["IGNITION_점화", "PRE_점화직전"])].head(5).copy()
    _cols = [
        "화재경보", "Code", "Name", "Market", "Theme", "최종점수", "구조점수",
        "종목수급점화점수", "테마점화점수", "거래대금20배", "ret1_%", "ret5_%", "ret20_%",
        "테마_확산도_%", "테마상대강도_%", "시총_억원", "추정유통시총_억원",
        "최대주주등지분율_%", "희석위험프록시_%", "구조데이터신뢰도", "선정이유"
    ]
    _rows = []
    for _, _r in _hot.iterrows():
        _rows.append({k: _tg_scalar(_r.get(k)) for k in _cols if k in _hot.columns})
    _payload = {
        "id": "lowfloat_fire",
        "mode": globals().get("DART_EFFECTIVE_MODE", "FULL" if globals().get("DART_API_KEY", "") else "LITE"),
        "total": int(len(result)),
        "counts": {str(k): int(v) for k, v in _counts.items()},
        "rows": _rows,
        "warnings": list(globals().get("LOWFLOAT_WARNINGS", [])),
    }
    print("__TG__" + _json.dumps(_payload, ensure_ascii=False))
except Exception as _e:
    print("__TG__" + _json.dumps({"id":"lowfloat_fire", "error": f"{type(_e).__name__}: {_e}"}, ensure_ascii=False))
